# Diffusion Transformer (DiT) Block with adaLN-Zero

**Difficulty**: 🔴 Hard

---

### Problem Statement

**DiT** (Peebles & Xie, *Scalable Diffusion Models with Transformers*, ICCV 2023) replaced the U-Net of latent diffusion with a plain Vision Transformer over patches of the VAE latent. The key design is how the conditioning vector $c$ (timestep embedding + class embedding) enters every block: **adaptive LayerNorm**. A small MLP regresses, from $c$, a per-sample *shift*, *scale* and *gate* for each residual branch; the LayerNorms themselves have no learnable affine parameters. In **adaLN-Zero** the regression layer is zero-initialized, so every gate starts at 0 and every block starts as the identity — the variant that won the paper's ablation against in-context tokens and cross-attention. DiT blocks (and their MMDiT descendants) are the backbone of PixArt, Stable Diffusion 3, Flux and Sora-style video models.

For each block, with $c \in \mathbb{R}^{B \times D}$ and tokens $x \in \mathbb{R}^{B \times N \times D}$:

$$(\beta_1, \gamma_1, \alpha_1, \beta_2, \gamma_2, \alpha_2) = \mathrm{Linear}(\mathrm{SiLU}(c)), \qquad \mathrm{modulate}(h, \beta, \gamma) = h \odot (1 + \gamma) + \beta$$

$$x \leftarrow x + \alpha_1 \odot \mathrm{Attn}\big(\mathrm{modulate}(\mathrm{LN}(x), \beta_1, \gamma_1)\big), \qquad x \leftarrow x + \alpha_2 \odot \mathrm{MLP}\big(\mathrm{modulate}(\mathrm{LN}(x), \beta_2, \gamma_2)\big)$$

where each $(B, D)$ vector is broadcast over the $N$ tokens of its sample. You will also write the timestep embedder and the `patchify` / `unpatchify` pair that turns latents into tokens and back.

---

### Requirements

1. **`modulate(x, shift, scale)`** — `x`: `(B, N, D)`, `shift`, `scale`: `(B, D)` → `x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)`.
2. **`TimestepEmbedder(hidden_size, freq_dim=256)`** — an `nn.Module` with
   - a `@staticmethod timestep_embedding(t, dim, max_period=10000)` → `(N, dim)` sinusoidal features of (possibly fractional or integer) timesteps `t` of shape `(N,)`: with $h$ = `dim // 2` and $P$ = `max_period`, frequencies $f_i = \exp(-\ln P \cdot i / h)$ for $i = 0, \dots, h-1$, arguments $a = t\,f$, and output `cat([cos(a), sin(a)], dim=-1)` (the DiT layout; assume `dim` even);
   - `forward(t)` = `Linear(freq_dim, hidden_size) → SiLU → Linear(hidden_size, hidden_size)` applied to `timestep_embedding(t, freq_dim)` → `(N, hidden_size)`.
3. **`DiTBlock(hidden_size, num_heads, mlp_ratio=4.0)`** with `forward(x, c)`, `x`: `(B, N, D)`, `c`: `(B, D)` → `(B, N, D)`:
   - `norm1`, `norm2` = `nn.LayerNorm(D, elementwise_affine=False, eps=1e-6)`;
   - multi-head self-attention over the $N$ tokens: `num_heads` heads of size `D // num_heads`, scores scaled by `1 / sqrt(head_dim)`, q/k/v and output projections with bias. Either `nn.MultiheadAttention(D, num_heads, batch_first=True)` or your own (a fused `Linear(D, 3D)` for q, k, v plus `Linear(D, D)` out, or four `Linear(D, D)`); no mask, no positional encoding inside the block;
   - MLP: `Linear(D, int(D * mlp_ratio)) → GELU(approximate="tanh") → Linear(int(D * mlp_ratio), D)`;
   - `self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(D, 6 * D))` with the Linear's **weight and bias initialized to zero**, chunked (`.chunk(6, dim=1)`) in the order `shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp`;
   - the two gated pre-norm residual updates above.
4. **`patchify(images, patch_size)`** — `(B, C, H, W)` → `(B, (H/p)(W/p), p·p·C)`. Patches in **row-major** order (token $k = i \cdot (W/p) + j$ holds pixels `[i·p:(i+1)·p, j·p:(j+1)·p]`); inside a token the features are ordered **(row-in-patch, col-in-patch, channel)** — the MAE / DiT layout.
5. **`unpatchify(tokens, patch_size, channels, h, w)`** — exact inverse of `patchify`; `h`, `w` are the image height and width **in pixels**.

---

### Constraints

- ✅ `nn.MultiheadAttention` (with `batch_first=True`) or `F.scaled_dot_product_attention` are fine.
- ✅ Assume `H` and `W` are divisible by `patch_size` and `hidden_size` by `num_heads`.
- ❌ No `timm`, `diffusers` or other model zoos.
- ❌ No learnable affine in the block's LayerNorms — adaLN supplies the scale and shift.
- ❌ No positional embedding or attention mask inside the block.

---

<details>
  <summary>💡 Hint</summary>

  - The conditioning is per *sample* but the activations are per *token*: every `(B, D)` modulation vector needs `unsqueeze(1)` to become `(B, 1, D)`.
  - Why `1 + scale`? With zero-initialized adaLN, `scale = 0` must leave the normalized activations untouched.
  - Sanity check: at initialization your block should return `x` **exactly**, whatever `c` is.
  - `patchify` is one `reshape` → `permute` → `reshape`; `unpatchify` runs the same three steps backwards. Write down the axis order `(B, C, H/p, p, W/p, p)` before permuting.

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

In [ ]:
def modulate(x, shift, scale):
    """x: (B, N, D); shift, scale: (B, D). Returns x * (1 + scale) + shift, broadcast over tokens."""
    # TODO
    raise NotImplementedError


class TimestepEmbedder(nn.Module):
    """Embeds scalar timesteps (N,) into (N, hidden_size) vectors."""

    def __init__(self, hidden_size, freq_dim=256):
        super().__init__()
        self.freq_dim = freq_dim
        # TODO: Linear(freq_dim, hidden_size) -> SiLU -> Linear(hidden_size, hidden_size)
        raise NotImplementedError

    @staticmethod
    def timestep_embedding(t, dim, max_period=10000):
        """t: (N,) float or long -> (N, dim) = cat([cos(t * f), sin(t * f)], -1),
        f_i = exp(-ln(max_period) * i / (dim // 2))."""
        # TODO
        raise NotImplementedError

    def forward(self, t):
        # TODO
        raise NotImplementedError


class DiTBlock(nn.Module):
    """Transformer block with adaLN-Zero conditioning."""

    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0):
        super().__init__()
        # TODO: norm1/norm2 (no affine, eps=1e-6), multi-head self-attention, MLP with GELU(tanh),
        #       self.adaLN_modulation = Sequential(SiLU, Linear(D, 6D)) with zero weight and bias
        raise NotImplementedError

    def forward(self, x, c):
        """x: (B, N, D) tokens, c: (B, D) conditioning -> (B, N, D)."""
        # TODO: chunk adaLN_modulation(c) into shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp
        raise NotImplementedError


def patchify(images, patch_size):
    """(B, C, H, W) -> (B, (H/p) * (W/p), p * p * C); row-major patches, features (row, col, channel)."""
    # TODO
    raise NotImplementedError


def unpatchify(tokens, patch_size, channels, h, w):
    """(B, (h/p) * (w/p), p * p * channels) -> (B, channels, h, w); exact inverse of patchify."""
    # TODO
    raise NotImplementedError

In [ ]:
# Test 1: modulate and the timestep embedder
x, shift, scale = torch.randn(2, 3, 4), torch.randn(2, 4), torch.randn(2, 4)
assert torch.allclose(modulate(x, shift, scale), x * (1 + scale[:, None]) + shift[:, None])

t = torch.tensor([0.0, 1.0, 250.5, 999.0])
emb = TimestepEmbedder.timestep_embedding(t, 8)
args = t[:, None] * torch.exp(-math.log(10000) * torch.arange(4) / 4)
assert emb.shape == (4, 8) and torch.allclose(emb, torch.cat([args.cos(), args.sin()], dim=-1), atol=1e-5)
te = TimestepEmbedder(32)
out = te(t)
assert out.shape == (4, 32) and not torch.allclose(out[0], out[1]), "different t -> different embeddings"
assert te(torch.tensor([3, 7])).shape == (2, 32), "integer timesteps must work"
print("Test 1 passed: modulate, timestep embedding")

In [ ]:
# Test 2: adaLN-Zero - identity at init, conditioning matters once the gates open, no positional info
torch.manual_seed(0)
blk = DiTBlock(32, num_heads=4)
x, c = torch.randn(2, 6, 32), torch.randn(2, 32)
assert torch.allclose(blk(x, c), x, atol=1e-6), "a freshly initialised DiT block must be the identity"

with torch.no_grad():                                   # pretend training moved the adaLN weights
    blk.adaLN_modulation[-1].weight.normal_(std=0.2)
    blk.adaLN_modulation[-1].bias.normal_(std=0.2)
out = blk(x, c)
assert out.shape == (2, 6, 32)
assert not torch.allclose(out, blk(x, c.flip(0)), atol=1e-4), "the output must depend on c"
perm = torch.randperm(6)
assert torch.allclose(blk(x[:, perm], c), out[:, perm], atol=1e-5), "the block must be permutation equivariant"
print("Test 2 passed: identity at init, conditioned, permutation equivariant")

In [ ]:
# Test 3: patchify / unpatchify
imgs = torch.randn(2, 3, 8, 12)
tok = patchify(imgs, 4)
assert tok.shape == (2, 6, 48), tok.shape
patch_11 = tok[:, 1 * 3 + 1].reshape(2, 4, 4, 3).permute(0, 3, 1, 2)   # token for patch row 1, col 1
assert torch.equal(patch_11, imgs[:, :, 4:8, 4:8]), "row-major patches, (row, col, channel) features"
assert torch.equal(unpatchify(tok, 4, 3, 8, 12), imgs), "unpatchify must invert patchify"
print("Test 3 passed: patchify / unpatchify")

In [ ]:
# Putting it together: a tiny DiT on 4-channel 16x16 "latents" (patch 2 -> 64 tokens)
class FinalLayer(nn.Module):
    """DiT's output head: adaLN shift/scale + a zero-initialised Linear to p*p*C_out."""

    def __init__(self, hidden, patch, out_ch):
        super().__init__()
        self.norm = nn.LayerNorm(hidden, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden, patch * patch * out_ch)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(hidden, 2 * hidden))
        for m in (self.linear, self.adaLN_modulation[-1]):
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)
        return self.linear(modulate(self.norm(x), shift, scale))


class TinyDiT(nn.Module):
    def __init__(self, in_ch=4, img=16, patch=2, hidden=64, depth=2, heads=4, num_classes=10):
        super().__init__()
        self.patch, self.in_ch, self.img = patch, in_ch, img
        self.x_embed = nn.Linear(patch * patch * in_ch, hidden)       # = Conv2d(k=s=patch) on patches
        self.pos = nn.Parameter(0.02 * torch.randn(1, (img // patch) ** 2, hidden))  # DiT: fixed 2-D sin-cos
        self.t_embed = TimestepEmbedder(hidden)
        self.y_embed = nn.Embedding(num_classes, hidden)
        self.blocks = nn.ModuleList(DiTBlock(hidden, heads) for _ in range(depth))
        self.final = FinalLayer(hidden, patch, in_ch)

    def forward(self, x, t, y):
        h = self.x_embed(patchify(x, self.patch)) + self.pos          # (B, N, hidden)
        c = self.t_embed(t) + self.y_embed(y)                          # (B, hidden): conditioning
        for blk in self.blocks:
            h = blk(h, c)
        return unpatchify(self.final(h, c), self.patch, self.in_ch, self.img, self.img)


torch.manual_seed(0)
net = TinyDiT()
x = torch.randn(8, 4, 16, 16)
t = torch.randint(0, 1000, (8,))
y = torch.randint(0, 10, (8,))
out = net(x, t, y)
print(f"output {tuple(out.shape)}, {sum(p.numel() for p in net.parameters()):,} parameters")
assert out.shape == x.shape and torch.all(out == 0), "an untrained DiT predicts exactly 0"

# Gradients reach the network progressively: at step 0 only the zero-initialised output layer
# gets a gradient; once it is non-zero, the blocks' adaLN gates start to receive one.
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
target = torch.randn_like(x)
for step in range(3):
    opt.zero_grad()
    F.mse_loss(net(x, t, y), target).backward()
    g_out = net.final.linear.weight.grad.abs().max().item()
    g_gate = net.blocks[0].adaLN_modulation[-1].weight.grad.abs().max().item()
    print(f"step {step}: max |grad| output layer {g_out:.1e}, block-0 adaLN {g_gate:.1e}")
    opt.step()

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("dit-block", modulate=modulate, TimestepEmbedder=TimestepEmbedder, DiTBlock=DiTBlock,
      patchify=patchify, unpatchify=unpatchify)